# 停牌与缺失处理的解析练习

本节接入C02、C03与C10。用没有证券身份的构造序列解释市场日对齐、滚动窗口、训练内填补和标签有效性。**所有数值均为解析功能演示，不是股票收益或真实行情，也不替代真实数据验收。**

当前股票协议采用Qlib启发的缺失值处理；它保留本课程80项特征、mRMR和角度缩放，并非运行完整Alpha158。原始缺失行情保持NA，处理发生在特征层。Qlib固定提交`54355232463878d2eebb91fe0ee5fa7fa1f5976c`。

先读[停牌与缺失教案](../../docs/停牌与缺失处理_qlib_missing_v1.md)。无需实验室数据、Qlib安装或量子云账户。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'课程大纲.md').is_file())


## 一条日历上的三种情况

市场休市不进入市场交易日索引；个股停牌可能在市场交易日中没有该股行情；来源漏数需要进一步查证，不能由NA本身判断原因。以下仅构造“观测缺失”，不声称识别了真实停牌。


In [ ]:
calendar = pd.RangeIndex(12, name='market_day')
observed = pd.Series([10., 11., np.nan, np.nan, 12., 13., 12., 14., 15., 13., 16., 17.], index=calendar)
original = observed.copy()
# 不向前填原始值；点差要求两个指定端点都有观测。
endpoint_change = observed / observed.shift(3) - 1
adjacent_return = observed / observed.shift(1) - 1
# Qlib普通Rolling的min_periods=1语义演示。实际各特征支持数另行保存。
rolling_mean = observed.rolling(3, min_periods=1).mean()
support = observed.notna().astype(int).rolling(3, min_periods=1).sum()
display(pd.DataFrame({'observed':observed,'window3_mean':rolling_mean,'window3_support':support,
                      'endpoint_change3':endpoint_change,'adjacent_return':adjacent_return}))
assert observed.equals(original)
assert np.isnan(endpoint_change.iloc[5])  # 原定端点缺失，不能寻找别的日期
assert support.iloc[4] == 1
assert adjacent_return.iloc[4] != adjacent_return.iloc[4]  # NaN，缺口后不是一日收益


## 填补的是特征，不是交易价格

训练中逐列有限值中位数是本协议的填补规则；Qlib的Fillna默认常数0只是一个可配置处理器，不意味着本课必须向原始价格填0。全缺失、常数、零IQR及重复特征先在训练资料中判定，不能从评价期资料选择保留哪些列。以下四列是无金融含义的矩阵。


In [ ]:
train_raw = np.array([[1., 2., np.nan, 5.], [2., np.nan, np.nan, 5.],
                      [3., 4., np.nan, 5.], [np.nan, 6., np.nan, 5.],
                      [5., 8., np.nan, 5.], [6., 10., np.nan, 5.]])
future_raw = np.array([[np.nan, 12., np.nan, 5.], [7., np.nan, np.nan, 5.]])
eligible = np.any(np.isfinite(train_raw), axis=0)
median = np.full(train_raw.shape[1], np.nan)
median[eligible] = np.nanmedian(train_raw[:, eligible], axis=0)
train_filled = np.where(np.isfinite(train_raw), train_raw, median)
iqr = np.full(train_raw.shape[1], np.nan)
iqr[eligible] = np.quantile(train_filled[:, eligible], .75, axis=0)-np.quantile(train_filled[:, eligible], .25, axis=0)
keep = eligible & (iqr > 0)
def apply_training_fill(raw):
    mask = ~np.isfinite(raw)
    filled = np.where(mask, median, raw)
    return filled[:, keep], mask[:, keep]
future_filled, future_missing = apply_training_fill(future_raw)
assert keep.tolist() == [True, True, False, False]
assert future_filled[0,0] == median[0]
assert future_missing.sum() == 2
assert np.isnan(future_raw[0,0])  # 输入原件未被修改
display(pd.DataFrame(future_filled, columns=['feature_0','feature_1']))
print('保留缺失掩码：', future_missing.tolist())


## 时间边界检查

目标保持原定次日开盘至第6个交易日开盘。端点缺失则标签无效；不延后端点，不填0。以下缩短为两个固定索引，只测试接口原则。未来矩阵改变不会重新拟合训练统计量。


In [ ]:
frozen_median = median.copy()
altered_future = future_raw.copy(); altered_future[0,1] = 1e9
_ = apply_training_fill(altered_future)
np.testing.assert_equal(median, frozen_median)
def endpoint_label(values, entry, exit):
    a,b = values.iloc[entry], values.iloc[exit]
    return np.nan if not (np.isfinite(a) and np.isfinite(b) and a>0 and b>0) else b/a-1
assert np.isnan(endpoint_label(observed, 2, 7))
assert endpoint_label(observed, 4, 7) == 14/12-1
# 训练标签结束日必须早于验证信号开始日。
train_signal = np.array([0,1,2,3]); train_label_end = train_signal + 6
validation_start = 8
assert train_signal[train_label_end < validation_start].tolist() == [0,1]
print('解析检查通过；云提交0；未读取真实行情。')


## 验收与作业

1. 解释为何滚动统计存在数值仍要报告有效支持数，不能把1个观测当作完整3日窗口。
2. 将缺失放在标签端点与特征窗口中部，说明两种情况为何采用不同处理。
3. 用测试证明评价期资料变化不影响训练中位数、mRMR或缩放对象；比较运行时只使用已经冻结的对象。

代码单元检查的是机制，不输出真实投资结论。正式课程结果见[版本索引](../../docs/QuKerNet先导实验版本索引.md)。

来源：[Rolling与min_periods](https://github.com/microsoft/qlib/blob/54355232463878d2eebb91fe0ee5fa7fa1f5976c/qlib/data/ops.py#L713-L755)、[Fillna](https://github.com/microsoft/qlib/blob/54355232463878d2eebb91fe0ee5fa7fa1f5976c/qlib/data/dataset/processor.py#L179-L193)、[DropnaLabel](https://github.com/microsoft/qlib/blob/54355232463878d2eebb91fe0ee5fa7fa1f5976c/qlib/data/dataset/processor.py#L105-L111)、[训练期归一化](https://github.com/microsoft/qlib/blob/54355232463878d2eebb91fe0ee5fa7fa1f5976c/qlib/data/dataset/processor.py#L262-L301)。
